In [ ]:
# Quick experiment rerun configuration

# Data selection
EXPERIMENT_TYPE = "synthetic"        # "synthetic" or "real"
EXPERIMENT_NAME = "gp_gaussian"      # synthetic examples: "gp_gaussian", "brownian_gaussian"; real examples: "bike", "cars", "cows", "spotify"
SPLIT_DATA_SEED = 0                  # the seed of the splitted dataset to load; (0, 1, 2, 3, 4)

FIXED_EFFECT_OPTION = "steps_1D"     # used only when EXPERIMENT_TYPE == "synthetic"

# Prediction scenario
PREDICTION_SCENARIO = "in_context"  # "in_context" or "few_shot"

# Models to rerun
MODEL_NAMES = ["gbm_group_cat"]  # e.g. "gbm_no_group", "gbm_group_cat", "lme", "gplinear", "np", "anp", "npboost", "anpboost"

# Config source
RUN_MODE = "best_model"  # "model_config" uses configs/model/*.yaml; "best_model" adds params from results/*/{model}_params.csv
VALIDATION_METRIC = "rmse" # used for config runs and to select params rows in best_model mode
PARAM_RESULTS_DIR = None   # None uses default: results/<EXPERIMENT_NAME>; otherwise set e.g. "results/gp_gaussian"

# Runtime behavior
DEVICE = "cpu"             # "cpu" or "cuda"
CUDA_VISIBLE_DEVICES = None # e.g. "0" when DEVICE == "cuda"; leave None to keep the current environment
WANDB_LOGGING = False
PLOT_INTERMEDIATE = False # Set to true if you want to plot intermediate model fits / coverages (only NP-based models do this)

# Extra Hydra overrides, appended last. Examples:
# ADDITIONAL_OVERRIDES = ["model.max_boosting_rounds=25", "wandb.project=test-reruns"]
ADDITIONAL_OVERRIDES = []

# Keep False to only print the commands. Set True for a rerun.
RUN_EXPERIMENTS = False

# Quick experiment rerun

This notebook is only a small launcher around `scripts/experiments/run_experiment.py`. It does not generate raw or split data.

Before running it, the selected data must already exist under `data/raw/...` and `data/split/...`.  
Generate it with `python -m scripts.data.generate_data ...` or with the matching bash scripts, for example `scripts/bash_scripts/synthetic_experiments/gp_gaussian/data_generation.sh` for synthetic data or `scripts/bash_scripts/real_experiments/spotify/generate_split_data.sh` for real data.


In [2]:
import math
import os
import re
import shlex
import subprocess
import sys
from pathlib import Path

import pandas as pd

from src.constants import TUNED_HYPERPARAMS_PER_MODEL
from src.data.data_layout import PROJECT_ROOT, RESULT_FOLDER, ADDITIONAL_RESULT_FOLDER

os.chdir(PROJECT_ROOT)
PROJECT_ROOT


WindowsPath('C:/npboost')

In [ ]:
VALID_DATASET_TYPES = {"real", "synthetic"}
VALID_TASKS = {"in_context", "few_shot"}
VALID_RUN_MODES = {"model_config", "best_model"}
NON_PARAM_COLUMNS = {
    "fixed_effect_type",
    "feature_dimension",
    "task_name",
    "selection_metric",
    "data_split_seed",
    "best_validation_score",
    "run_id",
    "test_rmse",
    "test_crps",
}


def normalize_task_name(task_name):
    normalized = task_name.replace("-", "_")
    if normalized not in VALID_TASKS:
        raise ValueError(f"Unknown prediction scenario: {task_name!r}. Expected one of {sorted(VALID_TASKS)}.")
    return normalized


def normalize_dataset_type(dataset_type):
    normalized = dataset_type.lower()
    if normalized not in VALID_DATASET_TYPES:
        raise ValueError(f"Unknown dataset type: {dataset_type!r}. Expected 'real' or 'synthetic'.")
    return normalized


def normalize_run_mode(run_mode):
    normalized = run_mode.lower()
    if normalized not in VALID_RUN_MODES:
        raise ValueError(f"Unknown run mode: {run_mode!r}. Expected one of {sorted(VALID_RUN_MODES)}.")
    return normalized


def fixed_effect_parts(fixed_effect_name):
    match = re.fullmatch(r"(.+)_([0-9]+)D", fixed_effect_name)
    if not match:
        raise ValueError(
            f"Cannot match {fixed_effect_name!r} to params CSV columns. "
            "Expected names like 'steps_1D', 'zero_2D', etc."
        )
    return match.group(1), int(match.group(2))


def hydra_bool(value):
    return str(bool(value)).lower()


def hydra_value(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    if isinstance(value, bool):
        return hydra_bool(value)
    return str(value)


def command_to_text(command):
    if os.name == "nt":
        return subprocess.list2cmdline(command)
    return shlex.join(command)


def candidate_param_paths(model_name):
    candidates = []
    if PARAM_RESULTS_DIR is not None:
        candidates.append(Path(PARAM_RESULTS_DIR))
    candidates.extend(
        [
            RESULT_FOLDER / EXPERIMENT_NAME,
            ADDITIONAL_RESULT_FOLDER / EXPERIMENT_NAME,
        ]
    )
    return [path / f"{model_name}_params.csv" for path in candidates]


def find_param_csv(model_name):
    for path in candidate_param_paths(model_name):
        if path.exists():
            return path
    searched = "\n".join(f"  - {path}" for path in candidate_param_paths(model_name))
    raise FileNotFoundError(f"No params CSV found for {model_name}. Searched:\n{searched}")


def select_best_params_row(model_name):
    dataset_type = normalize_dataset_type(EXPERIMENT_TYPE)
    task_name = normalize_task_name(PREDICTION_SCENARIO)
    params_path = find_param_csv(model_name)
    params = pd.read_csv(params_path)

    matches = params.copy()
    if "task_name" in matches.columns:
        matches = matches[matches["task_name"] == task_name]
    if "data_split_seed" in matches.columns:
        matches = matches[matches["data_split_seed"] == SPLIT_DATA_SEED]
    if "selection_metric" in matches.columns and VALIDATION_METRIC is not None:
        matches = matches[matches["selection_metric"] == VALIDATION_METRIC]

    if dataset_type == "synthetic" and {"fixed_effect_type", "feature_dimension"}.issubset(matches.columns):
        fixed_effect_type, feature_dimension = fixed_effect_parts(FIXED_EFFECT_OPTION)
        matches = matches[
            (matches["fixed_effect_type"] == fixed_effect_type)
            & (matches["feature_dimension"] == feature_dimension)
        ]

    if len(matches) == 0:
        raise ValueError(f"No params row matched {model_name} in {params_path}.")
    if len(matches) > 1:
        matches = matches.sort_values("best_validation_score") if "best_validation_score" in matches.columns else matches
        print(f"Multiple params rows matched {model_name}; using the first after sorting by best_validation_score.")

    return matches.iloc[0], params_path


def params_row_to_overrides(model_name, row):
    overrides = []

    if VALIDATION_METRIC is None and "selection_metric" in row and not pd.isna(row["selection_metric"]):
        overrides.append(f"model.validation_metric={row['selection_metric']}")

    for key in TUNED_HYPERPARAMS_PER_MODEL.get(model_name, []):
        value = row.get(key, None)
        value_text = hydra_value(value)
        if value_text is not None:
            overrides.append(f"{key}={value_text}")

    return overrides


In [ ]:
def base_overrides(model_name):
    dataset_type = normalize_dataset_type(EXPERIMENT_TYPE)
    task_name = normalize_task_name(PREDICTION_SCENARIO)
    overrides = [
        f"data={dataset_type}/{EXPERIMENT_NAME}",
        f"task={task_name}",
        f"model={model_name}",
        f"data.experiment.split_data_seed={SPLIT_DATA_SEED}",
        f"device={DEVICE}",
        f"wandb.enabled={hydra_bool(WANDB_LOGGING)}",
        f"++plot_intermediate={hydra_bool(PLOT_INTERMEDIATE)}",
    ]
    if dataset_type == "synthetic":
        overrides.insert(1, f"data/synthetic/fixed_effect={FIXED_EFFECT_OPTION}")
    if VALIDATION_METRIC is not None:
        overrides.append(f"model.validation_metric={VALIDATION_METRIC}")
    return overrides


def build_command(model_name):
    run_mode = normalize_run_mode(RUN_MODE)
    overrides = base_overrides(model_name)
    selected_row = None
    selected_params_path = None

    if run_mode == "best_model":
        selected_row, selected_params_path = select_best_params_row(model_name)
        overrides.extend(params_row_to_overrides(model_name, selected_row))

    overrides.extend(ADDITIONAL_OVERRIDES)
    command = [sys.executable, "-m", "scripts.experiments.run_experiment", *overrides]
    return command, selected_row, selected_params_path


commands = []
for model_name in MODEL_NAMES:
    command, selected_row, selected_params_path = build_command(model_name)
    commands.append(command)
    print(f"\nModel: {model_name}")
    if selected_params_path is not None:
        print(f"Params: {selected_params_path}")
        display_columns = [
            column
            for column in selected_row.index
            if column in {"run_id", "best_validation_score", "selection_metric", "task_name", "data_split_seed"}
            or column.startswith("model.")
            or column in {"best_boosting_round", "val_rmse_best_epoch"}
        ]
        display(selected_row[display_columns].to_frame("value"))
    print(command_to_text(command))


In [5]:
env = os.environ.copy()
if CUDA_VISIBLE_DEVICES is not None:
    env["CUDA_VISIBLE_DEVICES"] = str(CUDA_VISIBLE_DEVICES)

if not RUN_EXPERIMENTS:
    print("RUN_EXPERIMENTS=False, so the commands above were only previewed.")
else:
    for command in commands:
        print(f"\nRunning:\n{command_to_text(command)}\n")
        subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)
    print("\nAll requested experiment reruns finished.")


RUN_EXPERIMENTS=False, so the commands above were only previewed.
